In [1]:
import os
#os.environ['KAGGLE_USERNAME'] = "brycenweimingmanners"
#os.environ['KAGGLE_KEY'] = "KGAT_8a4296bf65fcb0961dccec671a6467a3"

In [2]:
import sys
from pathlib import Path
import pandas as pd
from src.data.merge import merge_nba_pipeline

In [3]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
sys.path.append(str(PROJECT_ROOT))

print(f"🎯 Root path added. Ready to import modules from: {PROJECT_ROOT}")

🎯 Root path added. Ready to import modules from: /Users/brycen/Documents/NTU/DAML/ML_final_projectv2/nba-salary-valuation


In [4]:
import download_data

# 直接執行下載邏輯
download_data.main()

# 設定原始檔案路徑
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
# 2. 精準讀取更新後的無損基礎表 (移除 combine_stats，加入全新的 biometrics)
stats_path = DATA_RAW / 'kaggle_nba_stats' / 'NBA Player Stats and Salaries_2010-2025.csv'
injury_path = DATA_RAW / 'kaggle_injury_history' / 'Injury_History.csv'
bio_path = DATA_RAW / 'kaggle_biometrics' / 'all_seasons.csv'

nba_stats_df = pd.read_csv(stats_path)
injury_df = pd.read_csv(injury_path)
bio_df = pd.read_csv(bio_path)

print(f"✅ Data loaded successfully!")
print(f"   -> NBA Stats Rows: {nba_stats_df.shape[0]:,}")
print(f"   -> Biometrics Rows: {bio_df.shape[0]:,}\n")

# 3. 呼叫整合流水線 (把舊的 combine_df 替換成全新的 bio_df)
print("⚙️ Running merge pipeline...")
final_dataset = merge_nba_pipeline(nba_stats_df, injury_df, bio_df)

print(f"🏆 Pipeline complete! Master dataset shape: {final_dataset.shape}")

2026-06-02 22:22:56,557 - INFO - ============================================================
2026-06-02 22:22:56,558 - INFO - 🚀 Phase 1: 環境建置 & 資料收集
2026-06-02 22:22:56,558 - INFO - ============================================================
2026-06-02 22:22:56,558 - INFO - ============================================================
2026-06-02 22:22:56,558 - INFO - 下載 Kaggle 薪資數據...
2026-06-02 22:22:56,559 - INFO - ============================================================


Dataset URL: https://www.kaggle.com/datasets/ratin21/nba-player-stats-and-salaries-2010-2025


2026-06-02 22:22:57,742 - INFO - ✅ 成功下載並解壓至: /Users/brycen/Documents/NTU/DAML/ML_final_projectv2/nba-salary-valuation/data/raw/kaggle_nba_stats
2026-06-02 22:22:57,750 - INFO -    📄 尋獲檔案: NBA Player Stats and Salaries_2010-2025.csv | 欄位形狀: 31 欄
2026-06-02 22:22:57,751 - INFO - ============================================================
2026-06-02 22:22:57,751 - INFO - 下載 Kaggle 傷病數據...
2026-06-02 22:22:57,751 - INFO - ============================================================


Dataset URL: https://www.kaggle.com/datasets/buyuknacar/active-nba-players-10-year-injury-history


2026-06-02 22:22:58,829 - INFO - ✅ 成功下載並解壓至: /Users/brycen/Documents/NTU/DAML/ML_final_projectv2/nba-salary-valuation/data/raw/kaggle_injury_history
2026-06-02 22:22:58,834 - INFO -    📄 尋獲檔案: Injury_History.csv | 欄位形狀: 5 欄
2026-06-02 22:22:58,836 - INFO - ============================================================
2026-06-02 22:22:58,837 - INFO - 下載 Kaggle NBA All-Player Biometrics 歷史數據...
2026-06-02 22:22:58,838 - INFO - ============================================================


Dataset URL: https://www.kaggle.com/datasets/justinas/nba-players-data


2026-06-02 22:23:00,110 - INFO - ✅ 成功下載並解壓至: /Users/brycen/Documents/NTU/DAML/ML_final_projectv2/nba-salary-valuation/data/raw/kaggle_biometrics
2026-06-02 22:23:00,114 - INFO -    📄 尋獲檔案: all_seasons.csv | 欄位形狀: 22 欄
2026-06-02 22:23:00,115 - INFO - ============================================================
2026-06-02 22:23:00,116 - INFO - 建立薪資帽歷史表 (2011-2027)...
2026-06-02 22:23:00,116 - INFO - ============================================================
2026-06-02 22:23:00,123 - INFO - ✅ 薪資帽歷史已存至 /Users/brycen/Documents/NTU/DAML/ML_final_projectv2/nba-salary-valuation/data/external/salary_cap_history.csv
2026-06-02 22:23:00,124 - INFO -    資料範圍: 2011-2027
2026-06-02 22:23:00,124 - INFO - ============================================================
2026-06-02 22:23:00,125 - INFO - 彙整資料清單 (data_inventory.md)...
2026-06-02 22:23:00,125 - INFO - ============================================================
2026-06-02 22:23:00,131 - INFO - ✅ 資料清單已生成: /Users/brycen/Documents/NTU/DAML/ML_

✅ Data loaded successfully!
   -> NBA Stats Rows: 7,298
   -> Biometrics Rows: 12,844

⚙️ Running merge pipeline...
🏆 Pipeline complete! Master dataset shape: (7298, 37)


In [5]:
# 1. 紀錄刪除前的大小的行數
rows_before = final_dataset.shape[0]

# 2. 執行刪除：只要該列包含任何一個 NaN 缺失值就整行剃除
# （這代表該球員必須同時具備：常規賽季數據、傷病追蹤紀錄、且當年有參與選秀體測）
clean_modeling_dataset = final_dataset.dropna()

# 3. 紀錄刪除後的大小的行數
rows_after = clean_modeling_dataset.shape[0]
rows_lost = rows_before - rows_after
retention_rate = (rows_after / rows_before) * 100

# --- 📊 輸出比較報告 ---
print("============================================================")
print("🎯 DROPPING MISSING ROWS: BEFORE VS AFTER REPORT")
print("============================================================")
print(f"原始資料總列數 (Before Drop):  {rows_before:,} rows")
print(f"清洗後剩餘列數 (After Drop):   {rows_after:,} rows")
print(f"遭剔除的資料列數 (Rows Lost):   {rows_lost:,} rows")
print(f"資料留存率 (Retention Rate):    {retention_rate:.2f}%")
print("============================================================")

# 4. 預覽最終乾淨的建模矩陣
print("\n🔥 Cleaned Master Dataset Preview:")
clean_modeling_dataset.head()

🎯 DROPPING MISSING ROWS: BEFORE VS AFTER REPORT
原始資料總列數 (Before Drop):  7,298 rows
清洗後剩餘列數 (After Drop):   6,330 rows
遭剔除的資料列數 (Rows Lost):   968 rows
資料留存率 (Retention Rate):    86.74%

🔥 Cleaned Master Dataset Preview:


,player,Salary,season,Pos,Age,Team,G,GS,MP,FG,...,BLK,TOV,PF,PTS,total_injuries,had_severe_injury,player_height,player_weight,draft_year,draft_number
0,kobe bryant,23034375,2010,SG,31,LAL,73,73,38.8,9.8,...,0.3,3.2,2.6,27.0,0,0,198.12,96.161504,1996,13
1,jermaine o'neal,23016000,2010,C,31,MIA,70,70,28.4,5.6,...,1.4,1.8,3.0,13.6,0,0,210.82,115.665960,1996,17
2,tracy mcgrady,22843124,2010,SG,30,NYK,30,24,22.4,3.0,...,0.5,1.5,1.3,8.2,0,0,203.20,102.058200,1997,9
3,tim duncan,22183220,2010,C,33,SAS,78,77,31.3,7.2,...,1.5,1.8,1.9,17.9,0,0,210.82,113.398000,1997,1
4,shaquille o'neal,21000000,2010,C,37,CLE,53,53,23.4,4.9,...,1.2,2.0,3.2,12.0,0,0,215.90,147.417400,1992,1
